# DSCI 550 — Assignment 5: Graph Neural Networks & Relational Graphs

**100 points · 12 questions**


| Part | Topic | Points |
|---|---|---:|
| 1 | Graph neighborhoods & receptive fields | 10 |
| 2 | Message passing from scratch | 20 |
| 3 | Neural graph models | 25 |
| 4 | Node classification & model comparison | 30 |
| 5 | Relational tables as graphs | 15 |
| **Total** |  | **100** |

## Important grading format

Each question asks you to implement a required function. Gradescope evaluates that function on **hidden graphs, tensors, or relational tables**, not only on the visible Cora example.

- Keep every required function name unchanged.
- Questions are graded independently; a mistake in one question does not automatically remove credit from later questions.
- Do not modify the provided setup cells.
- Do not add a final softmax to classification models. Cross-entropy expects logits.
- The autograder shows only the Q-level score/pass-fail result; it does not reveal correct answers or hidden-test details.

## Libraries

You will use:

- **PyTorch** for tensor operations, neural networks, backpropagation, and optimization.
- **PyTorch Geometric only to download/load Cora.**
- **Pandas** for the small relational-data section.

The GCN layer used in the assignment is provided in the setup. You do **not** need to install or use PyG convolution layers.


In [97]:
# PROVIDED SETUP — run this cell first. Do not modify.

import copy
import importlib.util
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# Used only to make a reproducible, class-stratified Cora split.
if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "scikit-learn"]
    )
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# PyTorch Geometric is used only to load Cora.
if importlib.util.find_spec("torch_geometric") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "torch-geometric"]
    )

from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

_cora_dataset = Planetoid(
    root="./data/Cora",
    name="Cora",
    split="public",  # loaded masks replaced below by 70/15/15
    transform=NormalizeFeatures(),
)
CORA_DATA = _cora_dataset[0]

# Custom 70/15/15 node-label split for this teaching assignment.
# This REPLACES the Planetoid public 140/500/1000 masks.
# Stratify by topic label and use a fixed seed for reproducibility.
# All graph nodes and edges remain available for GCN message passing.
def make_stratified_node_masks(y, seed=42):
    labels = y.detach().cpu().numpy()
    node_ids = np.arange(len(labels))
    n_train = int(round(0.70 * len(labels)))
    n_val = int(round(0.15 * len(labels)))

    train_ids, held_out_ids = train_test_split(
        node_ids,
        train_size=n_train,
        stratify=labels,
        random_state=seed,
    )
    val_ids, test_ids = train_test_split(
        held_out_ids,
        train_size=n_val,
        stratify=labels[held_out_ids],
        random_state=seed,
    )

    masks = []
    for indices in (train_ids, val_ids, test_ids):
        mask = torch.zeros(len(labels), dtype=torch.bool)
        mask[torch.as_tensor(indices, dtype=torch.long)] = True
        masks.append(mask)
    return tuple(masks)

CORA_DATA.train_mask, CORA_DATA.val_mask, CORA_DATA.test_mask = (
    make_stratified_node_masks(CORA_DATA.y, seed=RANDOM_STATE)
)

# Each node belongs to exactly one split; there are no unassigned nodes.
assert torch.all(
    CORA_DATA.train_mask.int()
    + CORA_DATA.val_mask.int()
    + CORA_DATA.test_mask.int()
    == 1
)

print(
    f"Cora: {CORA_DATA.num_nodes} nodes, "
    f"{CORA_DATA.num_edges} directed edge entries, "
    f"{CORA_DATA.num_node_features} features, "
    f"{_cora_dataset.num_classes} classes"
)
print(
    "Masks:",
    int(CORA_DATA.train_mask.sum()),
    "train /",
    int(CORA_DATA.val_mask.sum()),
    "validation /",
    int(CORA_DATA.test_mask.sum()),
    "test",
)


class GCNLayer(nn.Module):
    """
    PROVIDED GCN layer.

    edge_index has shape [2, E]: row 0 = sources, row 1 = destinations.
    The layer adds self-loops and applies symmetric degree normalization.
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.linear = nn.Linear(in_channels, out_channels, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x, edge_index):
        num_nodes = x.shape[0]
        src, dst = edge_index

        self_nodes = torch.arange(num_nodes, device=x.device)
        src = torch.cat([src, self_nodes])
        dst = torch.cat([dst, self_nodes])

        degree = torch.zeros(num_nodes, device=x.device, dtype=x.dtype)
        degree.index_add_(
            0,
            dst,
            torch.ones(dst.shape[0], device=x.device, dtype=x.dtype),
        )
        inv_sqrt_degree = degree.clamp(min=1).pow(-0.5)
        norm = inv_sqrt_degree[src] * inv_sqrt_degree[dst]

        transformed = self.linear(x)
        messages = transformed[src] * norm.unsqueeze(-1)

        out = torch.zeros(
            num_nodes,
            transformed.shape[1],
            device=x.device,
            dtype=x.dtype,
        )
        out.index_add_(0, dst, messages)
        return out + self.bias


CUSTOMERS = pd.DataFrame(
    {
        "customer_id": [101, 205, 330, 470, 590],
        "age": [24, 41, 35, 29, 52],
        "region_code": [0, 1, 1, 2, 0],
    }
)

PRODUCTS = pd.DataFrame(
    {
        "product_id": [11, 27, 46],
        "category_code": [0, 1, 0],
        "rating": [4.4, 4.8, 4.1],
    }
)

TRANSACTIONS = pd.DataFrame(
    {
        "transaction_id": [9001, 9002, 9003, 9004, 9005, 9006],
        "customer_id": [101, 205, 101, 330, 205, 470],
        "product_id": [11, 27, 46, 11, 46, 27],
        "amount": [32.0, 85.0, 18.0, 40.0, 51.0, 74.0],
        "quantity": [1, 2, 1, 1, 3, 2],
    }
)

TRANSACTION_FEATURES = torch.tensor(
    TRANSACTIONS[["amount", "quantity"]].to_numpy(),
    dtype=torch.float32,
)

Cora: 2708 nodes, 10556 directed edge entries, 1433 features, 7 classes
Masks: 1896 train / 406 validation / 406 test


## Part 1 — Graph Neighborhoods & Receptive Fields (10 points)

### Q1 — One-hop neighbors (5 pts)


Implement `one_hop_neighbors(edge_index, node_id)`.

Interpret each column of `edge_index` as a directed edge **source → destination**. Return a **sorted Python list of unique outgoing neighbors** of `node_id`.

If a self-loop exists, do not include `node_id` itself in the returned list.

Your function must work for arbitrary node IDs present in `edge_index`; do not hardcode anything about Cora.

In [98]:
def one_hop_neighbors(edge_index, node_id):
    # YOUR CODE HERE
    sources, destinations = edge_index[0], edge_index[1]

    neighbors = set()
    for i in range(len(sources)):
        src = sources[i]
        dst = destinations[i]
        
        if src == node_id and dst != node_id:
            neighbors.add(int(dst))

    return sorted(neighbors)

### Q2 — Two-hop receptive field (5 pts)


Implement `two_hop_receptive_field(edge_index, node_id)`.

Return a **sorted Python list of unique nodes reachable from `node_id` in one or two outgoing steps**, excluding `node_id` itself.

This represents the structural receptive field available to a two-layer message-passing model before considering self-loops.

In [99]:
def two_hop_receptive_field(edge_index, node_id):
    # YOUR CODE HERE
    first_hop = one_hop_neighbors(edge_index, node_id)
    
    
    receptive_field = set(first_hop)
    
    
    for neighbor in first_hop:
        second_hop = one_hop_neighbors(edge_index, neighbor)
        
        
        for node in second_hop:
            if node != node_id:
                receptive_field.add(node)
                
    
    return sorted(receptive_field)

## Part 2 — Message Passing from Scratch (20 points)

### Q3 — Mean neighbor aggregation (10 pts)


Implement `mean_neighbor_aggregate(x, edge_index)`.

For every node `v`, aggregate features from **incoming neighbors**. For an edge `u → v`, node `u` contributes its feature vector to node `v`.

Requirements:

- compute the coordinate-wise **mean** of incoming-neighbor features;
- do **not** add self-loops automatically;
- if a node has no incoming neighbors, return a zero vector for that node;
- return a tensor with the same shape as `x`.

Do not use a graph-learning library for this question; use PyTorch tensor operations.

In [100]:
def mean_neighbor_aggregate(x, edge_index):

    out_dtype = None
    if torch.is_floating_point(x):
        out_dtype = x.dtype

    else:
        out_dtype = torch.float32

    out = torch.zeros_like(x, dtype=out_dtype)
    
    num_nodes = x.size(0)
    counts = torch.zeros(num_nodes, dtype=out_dtype, device=x.device)
    
    sources = edge_index[0]
    destinations = edge_index[1]
    
    
    for i in range(edge_index.size(1)):
        u = sources[i]
        v = destinations[i]
        out[v] += x[u].to(out_dtype)
        counts[v] += 1
        
    
    for node in range(num_nodes):
        if counts[node] > 0:
            out[node] = out[node] / counts[node]
            
    return out

### Q4 — One message-passing update (10 pts)


Implement:

```python
message_passing_step(x, edge_index, W_self, W_neigh, b)
```

Use the mean incoming-neighbor representation from Q3 and compute

$$
h'_v =
\mathrm{ReLU}\left(
x_v W_{\mathrm{self}}
+
\mathrm{mean}_{u \in N(v)}(x_u) W_{\mathrm{neigh}}
+
b
\right).

$$

`W_self` and `W_neigh` have shape `[input_dim, output_dim]`, and `b` has shape `[output_dim]`.

The function must work for hidden graphs and hidden feature dimensions.

In [101]:
def message_passing_step(x, edge_index, W_self, W_neigh, b):
    # YOUR CODE HERE
    neigh_mean = mean_neighbor_aggregate(x, edge_index)
    
    x_float = x.to(W_self.dtype)
    neigh_float = neigh_mean.to(W_neigh.dtype)
    
    
    self_var = x_float @ W_self
    neigh_var = neigh_float @ W_neigh
    total = self_var + neigh_var + b
    
    return torch.relu(total)

## Part 3 — Neural Graph Models (25 points)

### Q5 — Node-wise MLP baseline (8 pts)


Implement:

```python
build_mlp(in_channels, hidden_channels, out_channels, dropout)
```

Return a PyTorch `nn.Module` with the architecture:

**Linear → ReLU → Dropout → Linear**

The model operates independently on each node and **ignores graph edges**. Its `forward` method must accept:

```python
model(x, edge_index=None)
```

and return one logit vector per node.

Do not apply softmax or log-softmax at the output.

In [102]:
def build_mlp(in_channels, hidden_channels, out_channels, dropout):
    class NodeMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.lin1 = nn.Linear(in_channels, hidden_channels)
            self.relu = nn.ReLU()
            self.dropout = nn.Dropout(p=dropout)
            self.lin2 = nn.Linear(hidden_channels, out_channels)

        def forward(self, x, edge_index=None):
            x = self.lin1(x)
            x = self.relu(x)
            x = self.dropout(x)
            x = self.lin2(x)
            return x

    return NodeMLP()

### Q6 — Two-layer GCN (12 pts)


Implement:

```python
build_gcn(in_channels, hidden_channels, out_channels, dropout)
```

Use the **provided `GCNLayer`** class. The required architecture is:

**GCNLayer → ReLU → Dropout → GCNLayer**

The returned model's forward method must accept:

```python
model(x, edge_index)
```

and return one logit vector per node.

Do not apply softmax or log-softmax at the output.

In [103]:
def build_gcn(in_channels, hidden_channels, out_channels, dropout):
    class TwoLayerGCN(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = GCNLayer(in_channels, hidden_channels)
            self.relu = nn.ReLU()
            self.dropout = nn.Dropout(p=dropout)
            self.conv2 = GCNLayer(hidden_channels, out_channels)

        def forward(self, x, edge_index):
            x = self.conv1(x, edge_index)
            x = self.relu(x)
            x = self.dropout(x)
            x = self.conv2(x, edge_index)
            return x

    return TwoLayerGCN()

### Q7 — Masked loss and accuracy (5 pts)


Implement `masked_loss_accuracy(logits, y, mask)`.

Use only entries selected by the Boolean `mask`.

Return exactly:

```python
loss, accuracy
```

where:

- `loss` is a scalar PyTorch tensor computed with cross-entropy;
- `accuracy` is a Python float in `[0, 1]`.

In [104]:
def masked_loss_accuracy(logits, y, mask):
    masked_logits = logits[mask]
    masked_y = y[mask]
    
    loss = F.cross_entropy(masked_logits, masked_y)
    
    preds = masked_logits.argmax(dim=-1)
    accuracy = (preds == masked_y).float().mean().item()
    
    return loss, accuracy

## Part 4 — Node Classification & Model Comparison (30 points)

### Q8 — One full-batch training epoch (8 pts)


Implement `train_one_epoch(model, data, optimizer)`.

For one training epoch:

1. put the model in training mode;
2. clear old gradients;
3. compute logits for **all nodes**;
4. compute cross-entropy using **only `data.train_mask`**;
5. backpropagate;
6. update the model with the supplied optimizer;
7. return the pre-update training loss as a Python float.

The function must work for both the MLP baseline and the GCN because both models use:

```python
model(data.x, data.edge_index)
```

In [105]:
def train_one_epoch(model, data, optimizer):
    # YOUR CODE HERE
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index)
    train_mask = data.train_mask.bool()
    loss = F.cross_entropy(logits[train_mask], data.y[train_mask])
    loss.backward()
    optimizer.step()
    
    return loss.item()

### Q9 — Validation and test evaluation (7 pts)


Implement `evaluate_node_model(model, data)`.

Evaluation must:

- use `model.eval()`;
- disable gradient tracking;
- compute validation loss using `data.val_mask`;
- compute validation accuracy using `data.val_mask`;
- compute test accuracy using `data.test_mask`;
- leave model parameters unchanged.

Return exactly:

```python
{
    "val_loss": float,
    "val_accuracy": float,
    "test_accuracy": float
}
```

In [106]:
def evaluate_node_model(model, data):
    model.eval()
    
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        
        
        val_mask = data.val_mask.bool()
        test_mask = data.test_mask.bool()
        
        val_loss = F.cross_entropy(logits[val_mask], data.y[val_mask]).item()
        
        
        preds = logits.argmax(dim=-1)
        val_accuracy = (preds[val_mask] == data.y[val_mask]).float().mean().item()
        test_accuracy = (preds[test_mask] == data.y[test_mask]).float().mean().item()
        
    return {
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "test_accuracy": test_accuracy,
    }

### Q10 — Train and compare the MLP and GCN (15 pts)


Implement:

```python
train_and_compare(data, max_epochs=200, patience=25)
```

Train both models using:

- hidden dimension: **32**
- dropout: **0.50**
- optimizer: **Adam**
- learning rate: **0.01**
- weight decay: **5e-4**
- reset `torch.manual_seed(42)` immediately before constructing each model;
- monitor **validation loss**;
- save the model state whenever validation loss is **strictly lower** than the previous best;
- stop after `patience` consecutive epochs without validation-loss improvement;
- restore the best state before reporting validation/test performance.

Derive `in_channels` and the number of classes from `data`; do not hardcode Cora's dimensions.

Return exactly:

```python
mlp_model, gcn_model, report
```

where `report` contains:

```python
{
    "mlp_epochs_trained": int,
    "gcn_epochs_trained": int,
    "mlp_val_accuracy": float,
    "mlp_test_accuracy": float,
    "gcn_val_accuracy": float,
    "gcn_test_accuracy": float,
    "gcn_minus_mlp_test_accuracy": float
}
```


In [107]:
def train_and_compare(data, max_epochs=200, patience=25):
    in_channels = data.x.size(1)
    if hasattr(data, "num_classes") and data.num_classes is not None:
        out_channels = int(data.num_classes)
    else:
        out_channels = int(data.y.max().item()) + 1

    hidden_channels = 32
    dropout = 0.50
    lr = 0.01
    weight_decay = 5e-4

    def run_training(build_fn):
        torch.manual_seed(42)
        model = build_fn(in_channels, hidden_channels, out_channels, dropout)
        model = model.to(data.x.device)

        optimizer = torch.optim.Adam(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )

        best_val_loss = float("inf")
        best_state = None
        epochs_without_improvement = 0
        epochs_trained = 0

        for i in range(max_epochs):
            train_one_epoch(model, data, optimizer)
            epochs_trained += 1

            metrics = evaluate_node_model(model, data)
            val_loss = metrics["val_loss"]

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epochs_without_improvement >= patience:
                break

        if best_state is not None:
            model.load_state_dict(best_state)
        final_metrics = evaluate_node_model(model, data)

        return model, epochs_trained, final_metrics

    mlp_model, mlp_epochs, mlp_metrics = run_training(build_mlp)
    gcn_model, gcn_epochs, gcn_metrics = run_training(build_gcn)

    report = {
        "mlp_epochs_trained": int(mlp_epochs),
        "gcn_epochs_trained": int(gcn_epochs),
        "mlp_val_accuracy": float(mlp_metrics["val_accuracy"]),
        "mlp_test_accuracy": float(mlp_metrics["test_accuracy"]),
        "gcn_val_accuracy": float(gcn_metrics["val_accuracy"]),
        "gcn_test_accuracy": float(gcn_metrics["test_accuracy"]),
        "gcn_minus_mlp_test_accuracy": float(
            gcn_metrics["test_accuracy"] - mlp_metrics["test_accuracy"]
        ),
    }

    return mlp_model, gcn_model, report

In [108]:
# PROVIDED CORA TRAINING / REPORT CELL — run after completing Q5-Q10.
# Do not modify this cell.

CORA_MLP, CORA_GCN, CORA_REPORT = train_and_compare(CORA_DATA)

print("Cora node-classification results")
print("--------------------------------")
print(f"MLP epochs trained:      {CORA_REPORT['mlp_epochs_trained']}")
print(f"GCN epochs trained:      {CORA_REPORT['gcn_epochs_trained']}")
print(f"MLP validation accuracy: {CORA_REPORT['mlp_val_accuracy']:.4f}")
print(f"MLP test accuracy:       {CORA_REPORT['mlp_test_accuracy']:.4f}")
print(f"GCN validation accuracy: {CORA_REPORT['gcn_val_accuracy']:.4f}")
print(f"GCN test accuracy:       {CORA_REPORT['gcn_test_accuracy']:.4f}")
print(
    "GCN - MLP test accuracy:",
    f"{CORA_REPORT['gcn_minus_mlp_test_accuracy']:+.4f}"
)

Cora node-classification results
--------------------------------
MLP epochs trained:      200
GCN epochs trained:      200
MLP validation accuracy: 0.7241
MLP test accuracy:       0.7709
GCN validation accuracy: 0.8448
GCN test accuracy:       0.8818
GCN - MLP test accuracy: +0.1108


## Part 5 — Relational Tables as Graphs (15 points)

The paper *Relational Deep Learning: Graph Representation Learning on Relational Databases* proposes viewing a relational database as a heterogeneous graph:

- a **row** becomes a node;
- the **table** determines the node type;
- primary/foreign-key relationships become edges;
- message passing can propagate information across those relationships.

The following two questions use the supplied `CUSTOMERS`, `TRANSACTIONS`, and `PRODUCTS` tables as a small version of this idea.

### Q11 — Convert primary/foreign-key relationships into edges (7 pts)


Implement:

```python
build_relational_edges(customers, transactions, products)
```

Use **row positions** as graph-local node indices, not the raw database IDs.

Return:

```python
{
    "customer_transaction": customer_transaction_edge_index,
    "product_transaction": product_transaction_edge_index
}
```

Requirements:

- `customer_transaction_edge_index.shape == (2, number_of_transactions)`
  - row 0: customer row index
  - row 1: transaction row index
- `product_transaction_edge_index.shape == (2, number_of_transactions)`
  - row 0: product row index
  - row 1: transaction row index
- edges follow the current row order of `transactions`;
- do not assume that primary keys are contiguous, sorted, or start at zero.

In [109]:
def build_relational_edges(customers, transactions, products):
    cust_pk_col = None
    for col in customers.columns:
        if "customer" in col.lower() and "id" in col.lower():
            cust_pk_col = col
            break

    prod_pk_col = None
    for col in products.columns:
        if "product" in col.lower() and "id" in col.lower():
            prod_pk_col = col
            break

    tx_cust_fk_col = None
    tx_prod_fk_col = None
    for col in transactions.columns:
        col_lower = col.lower()
        if "customer" in col_lower and "id" in col_lower:
            tx_cust_fk_col = col
        elif "product" in col_lower and "id" in col_lower:
            tx_prod_fk_col = col

    cust_id_to_pos = {}
    for pos, cid in enumerate(customers[cust_pk_col].tolist()):
        cust_id_to_pos[cid] = pos

    prod_id_to_pos = {}
    for pos, pid in enumerate(products[prod_pk_col].tolist()):
        prod_id_to_pos[pid] = pos

    cust_src = []
    prod_src = []
    tx_dst = []

    tx_cust_ids = transactions[tx_cust_fk_col].tolist()
    tx_prod_ids = transactions[tx_prod_fk_col].tolist()

    for tx_pos in range(len(transactions)):
        cid = tx_cust_ids[tx_pos]
        pid = tx_prod_ids[tx_pos]

    

        cust_src.append(cust_id_to_pos[cid])
        prod_src.append(prod_id_to_pos[pid])
        tx_dst.append(tx_pos)

    if len(tx_dst) == 0:
        customer_transaction_edge_index = torch.empty((2, 0), dtype=torch.long)
        product_transaction_edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        customer_transaction_edge_index = torch.tensor([cust_src, tx_dst], dtype=torch.long)
        product_transaction_edge_index = torch.tensor([prod_src, tx_dst], dtype=torch.long)

    return {
        "customer_transaction": customer_transaction_edge_index,
        "product_transaction": product_transaction_edge_index,
    }



### Q12 — Aggregate transaction messages into customer nodes (8 pts)


Implement:

```python
aggregate_transactions_to_customers(
    transaction_x,
    customer_transaction_edge_index,
    num_customers
)
```

`transaction_x` contains transaction feature vectors. In `customer_transaction_edge_index`:

- row 0 contains customer row indices;
- row 1 contains transaction row indices.

For every customer, return the **mean feature vector of that customer's transactions**.

Requirements:

- output shape: `[num_customers, transaction_feature_dim]`;
- customers with no transactions receive a zero vector;
- use PyTorch tensor operations;
- do not assume edges are sorted.

This is a simplified message-passing analogue of a relational `JOIN + GROUP BY + MEAN` operation.

In [110]:
def aggregate_transactions_to_customers(
    transaction_x,
    customer_transaction_edge_index,
    num_customers,
):
    # YOUR CODE HERE
    out_dtype = None
    if torch.is_floating_point(transaction_x):
        out_dtype = transaction_x.dtype

    else:
        out_dtype = torch.float32
    
    device = transaction_x.device
    feature_dim = transaction_x.size(1)

    sums = torch.zeros((num_customers, feature_dim), dtype=out_dtype, device=device)
    counts = torch.zeros(num_customers, dtype=out_dtype, device=device)

    cust_idx = customer_transaction_edge_index[0].long()
    tx_idx = customer_transaction_edge_index[1].long()

    if cust_idx.numel() > 0:
        selected_tx = transaction_x[tx_idx].to(out_dtype)
        sums.index_add_(0, cust_idx, selected_tx)

        ones = torch.ones(cust_idx.size(0), dtype=out_dtype, device=device)
        counts.index_add_(0, cust_idx, ones)

    safe_counts = counts.clamp(min=1.0).unsqueeze(1)
    return sums / safe_counts

### Part 5 — Run the complete relational graph pipeline

The two functions above should not just be defined; **use them together**. The provided cell below performs an end-to-end relational-data experiment:

1. Turn the primary/foreign-key relationships into customer→transaction and product→transaction graph edges (Q11).
2. Aggregate transaction feature vectors into customer feature vectors using the resulting graph edges (Q12).
3. Show the resulting customer features and product transaction counts.
4. Verify that graph message aggregation agrees with a familiar `JOIN / GROUP BY / MEAN`-style Pandas computation.

Run this cell after completing Q11 and Q12. Do not modify this provided demonstration cell.

In [111]:
# PROVIDED FULL PART 5 RUN — run after implementing Q11 and Q12.
# This demonstrates the pipeline on the sample relational database.

# Step 1: Build edges from primary/foreign-key relationships.
RELATIONAL_EDGES = build_relational_edges(CUSTOMERS, TRANSACTIONS, PRODUCTS)
customer_transaction_edges = RELATIONAL_EDGES["customer_transaction"]
product_transaction_edges = RELATIONAL_EDGES["product_transaction"]

print("=== Relational graph construction ===")
print(f"Customer nodes:    {len(CUSTOMERS)}")
print(f"Transaction nodes: {len(TRANSACTIONS)}")
print(f"Product nodes:     {len(PRODUCTS)}")
print("Customer -> transaction edge_index [customer row; transaction row]:")
print(customer_transaction_edges)
print("Product -> transaction edge_index [product row; transaction row]:")
print(product_transaction_edges)

# Step 2: Send transaction feature messages to linked customer nodes.
customer_messages = aggregate_transactions_to_customers(
    TRANSACTION_FEATURES,
    customer_transaction_edges,
    num_customers=len(CUSTOMERS),
)

# Step 3: Interpret the graph-aggregated features as a table.
CUSTOMER_GRAPH_FEATURES = CUSTOMERS[["customer_id"]].copy()
CUSTOMER_GRAPH_FEATURES["mean_amount"] = customer_messages[:, 0].detach().cpu().numpy()
CUSTOMER_GRAPH_FEATURES["mean_quantity"] = customer_messages[:, 1].detach().cpu().numpy()

print("\n=== Customer features learned from neighbor aggregation ===")
print(CUSTOMER_GRAPH_FEATURES.to_string(index=False))

PRODUCT_TRANSACTION_COUNTS = torch.bincount(
    product_transaction_edges[0], minlength=len(PRODUCTS)
)
PRODUCT_GRAPH_SUMMARY = PRODUCTS[["product_id"]].copy()
PRODUCT_GRAPH_SUMMARY["transaction_count"] = PRODUCT_TRANSACTION_COUNTS.tolist()
print("\n=== Product graph neighborhoods ===")
print(PRODUCT_GRAPH_SUMMARY.to_string(index=False))

# Step 4: Verify the message-passing result against a database aggregation.
# Reindex also accounts for customers with no transactions (fill with zero).
pandas_reference = (
    TRANSACTIONS.groupby("customer_id")[["amount", "quantity"]]
    .mean()
    .reindex(CUSTOMERS["customer_id"])
    .fillna(0.0)
    .to_numpy(dtype=np.float32)
)
np.testing.assert_allclose(
    customer_messages.detach().cpu().numpy(),
    pandas_reference,
    atol=1e-5,
    rtol=1e-5,
)
print("\nVerified: graph neighbor mean matches Pandas GROUP BY mean.")
print("Verified: a customer with no transactions has a zero vector.")
assert torch.all(customer_messages[-1] == 0)

=== Relational graph construction ===
Customer nodes:    5
Transaction nodes: 6
Product nodes:     3
Customer -> transaction edge_index [customer row; transaction row]:
tensor([[0, 1, 0, 2, 1, 3],
        [0, 1, 2, 3, 4, 5]])
Product -> transaction edge_index [product row; transaction row]:
tensor([[0, 1, 2, 0, 2, 1],
        [0, 1, 2, 3, 4, 5]])

=== Customer features learned from neighbor aggregation ===
 customer_id  mean_amount  mean_quantity
         101         25.0            1.0
         205         68.0            2.5
         330         40.0            1.0
         470         74.0            2.0
         590          0.0            0.0

=== Product graph neighborhoods ===
 product_id  transaction_count
         11                  2
         27                  2
         46                  2

Verified: graph neighbor mean matches Pandas GROUP BY mean.
Verified: a customer with no transactions has a zero vector.


---
## Submission

Before submitting:

1. restart the kernel;
2. run the notebook from top to bottom;
3. make sure all 12 required functions are defined;
4. run the provided Cora training/report cell;
5. run the provided Part 5 relational graph demonstration cell;
6. save the notebook with both outputs, then submit the `.ipynb` file to Gradescope.
